# Fine-tuning SLM — Chatbot Tim Legal berbasis RAG
**Proyek Akhir PGABL — Vika Putri Ariyanti**

Notebook ini memenuhi **Kriteria 1** sampai level **Skilled**, dan menyiapkan model
instruct yang akan dilanjutkan ke tahap GRPO (level Advanced) pada notebook terpisah.

| Kode | Ketentuan rubrik | Sel |
|------|------------------|-----|
| F1.1 | Mapping dataset ke Chat Template + print baris sebelum & sesudah | §3 |
| F1.2 | QLoRA 4-bit + **double quantization**, LoRA pada MHA **dan** FFN | §5 |
| F1.3 | `SFTTrainer` ≥ **800 steps** tanpa OOM | §6 |
| F1.4 | `push_to_hub_merged(..., save_method="merged_16bit")` | §8 |
| F1.5 | Split train/validation + `eval_strategy="steps"` + logging | §4, §6 |
| F1.6 | **Dua** eksperimen hyperparameter + perbandingan kurva loss | §6, §7 |

**Dataset (wajib):** `Ichsan2895/alpaca-gpt4-indonesian`
**Base model (wajib text generation, didukung Unsloth):** `unsloth/Qwen2.5-3B-bnb-4bit`
— model *base* (bukan instruct) dari repo kuantisasi resmi Unsloth, sehingga kemampuan
mengikuti instruksi benar-benar berasal dari fine-tuning yang kita lakukan sendiri.

## 1. Instalasi dependensi

In [1]:
# Unsloth 2026.1.1 sendiri mensyaratkan transformers>=4.51.3, trl>=0.18.2,
# datasets>=3.4.1, bitsandbytes>=0.45.5 — pin lama (trl 0.13 dst.) membuat pip gagal.
# trl dikunci di 0.18.2 karena SFTConfig(max_seq_length=...) dihapus pada trl 0.20+.
# peft>=0.18 wajib: unsloth 2026.1.1 mengoper argumen `ensure_weight_tying` ke
# LoraConfig, yang baru ada sejak peft 0.18 (peft 0.17 -> TypeError).
# torch sengaja TIDAK dipin: pakai bawaan Colab, biar unsloth menarik xformers yang cocok.
!pip install -q -U "unsloth==2026.1.1" "trl==0.18.2" "peft==0.18.0" \
    "transformers==4.57.1" "datasets==3.6.0" "accelerate==1.10.1" \
    "bitsandbytes==0.47.0" "wandb==0.19.4"

import torch, transformers, trl, datasets, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("trl         :", trl.__version__)
print("datasets    :", datasets.__version__)
print("peft        :", peft.__version__)
print("GPU         :", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.2/379.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

torch       : 2.11.0+cu128
transformers: 4.57.1
trl         : 0.18.2
datasets    : 3.6.0
peft        : 0.18.0
GPU         : Tesla T4


## 2. Konfigurasi & kredensial

Token **tidak pernah** ditulis sebagai literal di notebook (larangan rubrik no. 9).
Simpan `HF_TOKEN` dan `WANDB_API_KEY` di panel **Secrets** Colab (ikon kunci), lalu
notebook membacanya sebagai environment variable.

In [2]:
import os

def load_secret(name, required=True):
    # Prioritas: Colab Secrets -> environment variable yang sudah ada.
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise RuntimeError(
            f"{name} belum di-set. Tambahkan lewat panel Secrets Colab, jangan hard-code."
        )
    if value:
        os.environ[name] = value
    # Sengaja hanya mencetak status, bukan nilainya.
    print(f"{name}: {'tersedia' if value else 'tidak tersedia'}")
    return value

HF_TOKEN = load_secret("HF_TOKEN")
WANDB_KEY = load_secret("WANDB_API_KEY", required=False)

HF_TOKEN: tersedia
WANDB_API_KEY: tersedia


In [3]:
# ---- Konfigurasi global ----
# Jika T4 gratis terasa terlalu lambat atau kehabisan kuota, ganti ke varian 1.5B:
#   BASE_MODEL = "unsloth/Qwen2.5-1.5B-bnb-4bit"
# Keduanya sama-sama Small Language Model base yang didukung Unsloth dan memenuhi
# rubrik; yang 1.5B kira-kira dua kali lebih cepat per step.
BASE_MODEL     = "unsloth/Qwen2.5-3B-bnb-4bit"   # base text-generation, didukung Unsloth
DATASET_ID     = "Ichsan2895/alpaca-gpt4-indonesian"
MAX_SEQ_LENGTH = 1024
HF_USERNAME    = "vikaputri"                      # <-- ganti dengan username HF Anda
HF_REPO_SFT    = f"{HF_USERNAME}/qwen2.5-3b-legal-id-sft"

SEED = 3407

# W&B bersifat OPSIONAL. Kalau key tidak ada / tidak valid (mis. kurang dari 40
# karakter karena kepotong saat copy), login-nya dibiarkan gagal secara halus dan
# training tetap jalan tanpa logging W&B (report_to="none") — bukan error fatal.
USE_WANDB = bool(WANDB_KEY)
if USE_WANDB:
    try:
        import wandb
        wandb.login(key=WANDB_KEY, verify=True)
        os.environ["WANDB_PROJECT"] = "pgabl-legal-chatbot"
    except Exception as e:
        print("W&B dinonaktifkan (login gagal):", e)
        USE_WANDB = False
REPORT_TO = "wandb" if USE_WANDB else "none"
print("logging ke:", REPORT_TO)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


W&B dinonaktifkan (login gagal): API key must be 40 characters long, yours was 37
logging ke: none


### 2.1 Lokasi checkpoint

Sesi Colab gratis dapat terputus di tengah training. Dengan me-mount Google Drive,
checkpoint dan adapter hasil tiap eksperimen selamat, sehingga training bisa
dilanjutkan (`resume_from_checkpoint`) alih-alih diulang dari nol.

In [4]:
CKPT_ROOT = "outputs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_ROOT = "/content/drive/MyDrive/pgabl_outputs"
except Exception:
    print("Drive tidak tersedia — checkpoint disimpan di disk sesi (hilang bila terputus).")

os.makedirs(CKPT_ROOT, exist_ok=True)
print("checkpoint disimpan di:", CKPT_ROOT)

Mounted at /content/drive
checkpoint disimpan di: /content/drive/MyDrive/pgabl_outputs


## 3. Dataset & mapping ke Chat Template — **F1.1**

Dataset mentah berbasis **Alpaca-GPT4** dengan kolom `input` (instruksi + konteks yang
sudah tergabung) dan `output`. Kita ubah ke standar **ChatML** milik Qwen2.5 memakai
`tokenizer.apply_chat_template` lewat fungsi `datasets.map`, lalu cetak satu baris
**sebelum** dan **sesudah** mapping.

In [5]:
from datasets import load_dataset

raw = load_dataset(DATASET_ID, split="train")
print(raw)
print("\nKolom:", raw.column_names)

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset({
    features: ['Unnamed: 0', 'input', 'output'],
    num_rows: 49969
})

Kolom: ['Unnamed: 0', 'input', 'output']


In [6]:
# ---- Contoh baris SEBELUM mapping (masih format Alpaca mentah) ----
sample_raw = raw[0]
print("=" * 80)
print("SEBELUM MAPPING (format Alpaca mentah)")
print("=" * 80)
for k, v in sample_raw.items():
    print(f"[{k}] {v}")

SEBELUM MAPPING (format Alpaca mentah)
[Unnamed: 0] 1
[input] Saranlah slogan untuk kampanye daur ulang

[output] 1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."


In [7]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer

# Tokenizer dimuat lebih dulu (tanpa bobot model) supaya proses mapping dataset
# tidak perlu menunggu model dan tidak memakan VRAM.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

SYSTEM_PROMPT = (
    "Anda adalah asisten AI Tim Legal yang menjawab pertanyaan secara akurat, "
    "ringkas, dan selalu menggunakan Bahasa Indonesia."
)

# Dataset ini menggabungkan instruksi+konteks ke dalam satu kolom `input`
# (kolom yang tersedia hanya: 'Unnamed: 0', 'input', 'output') — tidak ada
# kolom `instruction` terpisah. Jadi `input` -> pesan user, `output` -> asisten.
def to_chat_template(batch):
    texts = []
    for user_content, output in zip(batch["input"], batch["output"]):
        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": output},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

dataset = raw.map(
    to_chat_template,
    batched=True,
    remove_columns=raw.column_names,
    desc="Mapping Alpaca -> ChatML",
)
print(dataset)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/tmp/ipykernel_391/4129404594.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth Zoo will now patch everything to make training faster!


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

Unsloth: Will map <|im_end|> to EOS = <|endoftext|>.


Mapping Alpaca -> ChatML:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 49969
})


In [8]:
# ---- Contoh baris SESUDAH mapping (lengkap dengan token spesial ChatML) ----
print("=" * 80)
print("SESUDAH MAPPING (Chat Template ChatML)")
print("=" * 80)
print(dataset[0]["text"])
print("=" * 80)
print("Token spesial yang muncul:",
      [t for t in ["<|im_start|>", "<|im_end|>"] if t in dataset[0]["text"]])

SESUDAH MAPPING (Chat Template ChatML)
<|im_start|>system
Anda adalah asisten AI Tim Legal yang menjawab pertanyaan secara akurat, ringkas, dan selalu menggunakan Bahasa Indonesia.<|im_end|>
<|im_start|>user
Saranlah slogan untuk kampanye daur ulang
<|im_end|>
<|im_start|>assistant
1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."<|im_end|>

Token spesial yang muncul: ['<|im_start|>', '<|im_end|>']


## 4. Panjang urutan, split train / validation — **F1.5**

Sebelum dibagi, dataset disaring dari contoh yang melebihi `MAX_SEQ_LENGTH`.
Unsloth memotong `input_ids` tetapi `labels` ikut tidak terpotong, sehingga contoh
panjang membuat `cross_entropy` gagal dengan shape mismatch di tengah training.


In [9]:
# ---- Saring contoh yang melebihi MAX_SEQ_LENGTH ----
# Unsloth memotong `input_ids` menjadi MAX_SEQ_LENGTH tetapi `labels` TIDAK ikut
# terpotong, sehingga contoh yang kepanjangan memicu shape mismatch di
# cross_entropy (input 1024 vs target 1092) dan training gagal di tengah jalan.
#
# Menyaring di sini lebih baik daripada mengandalkan truncation trainer: memotong
# contoh di tengah kalimat justru mengajari model menghasilkan jawaban terpotong.
# Dari 49.969 contoh hanya 20.000 yang dipakai, jadi membuang yang panjang tidak
# mengurangi jatah data sama sekali.
#
# Margin dipakai karena tokenisasi di dalam trainer masih dapat menambah token
# penutup, sehingga contoh yang persis di batas tetap bisa melewati limit.
LENGTH_MARGIN = 8
BATAS_TOKEN   = MAX_SEQ_LENGTH - LENGTH_MARGIN

def _cukup_pendek(batch):
    ids = tokenizer(batch["text"], add_special_tokens=False)["input_ids"]
    return [len(x) <= BATAS_TOKEN for x in ids]

sebelum = len(dataset)
dataset = dataset.filter(_cukup_pendek, batched=True,
                         desc=f"Saring <= {BATAS_TOKEN} token")
print(f"dibuang   : {sebelum - len(dataset)} contoh melebihi {BATAS_TOKEN} token")
print(f"tersisa   : {len(dataset)} contoh")

# Subset dipakai agar durasi training realistis di GPU free-tier,
# tetap jauh di atas kebutuhan 800 steps.
SUBSET_SIZE = 20_000
dataset = dataset.shuffle(seed=SEED).select(range(min(SUBSET_SIZE, len(dataset))))

splits = dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset, eval_dataset = splits["train"], splits["test"]
print("train     :", len(train_dataset))
print("validation:", len(eval_dataset))

# Bukti bahwa penyaringan benar-benar bekerja — kalau angka ini masih melebihi
# MAX_SEQ_LENGTH, training pasti gagal lagi dengan error yang sama.
maks = max(len(x) for x in
           tokenizer(train_dataset["text"], add_special_tokens=False)["input_ids"])
print(f"panjang token maksimum di train: {maks} (limit {MAX_SEQ_LENGTH})")
assert maks <= MAX_SEQ_LENGTH, "Masih ada contoh melebihi MAX_SEQ_LENGTH."


Saring <= 1016 token:   0%|          | 0/49969 [00:00<?, ? examples/s]

dibuang   : 16 contoh melebihi 1016 token
tersisa   : 49953 contoh
train     : 19000
validation: 1000
panjang token maksimum di train: 1016 (limit 1024)


## 5. Memuat model dengan QLoRA — **F1.2**

- `load_in_4bit=True` dengan **double quantization** aktif (`bnb_4bit_use_double_quant=True`).
- Adapter LoRA dipasang pada **dua komponen komputasi utama secara penuh**:
  - *Multi-Head Attention*: `q_proj`, `k_proj`, `v_proj`, `o_proj`
  - *Feed Forward Network*: `gate_proj`, `up_proj`, `down_proj`

Model dibungkus dalam fungsi agar dapat dibuat ulang bersih untuk tiap eksperimen (F1.6).

In [10]:
from transformers import BitsAndBytesConfig

ATTENTION_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]   # Multi-Head Attention
FFN_MODULES       = ["gate_proj", "up_proj", "down_proj"]      # Feed Forward Network
TARGET_MODULES    = ATTENTION_MODULES + FFN_MODULES

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,          # <-- double quantization (wajib)
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

def build_model(lora_r, lora_alpha, lora_dropout=0.0):
    kwargs = dict(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,                # auto: bfloat16 bila didukung
        load_in_4bit=True,
        token=HF_TOKEN,
    )
    try:
        model, tok = FastLanguageModel.from_pretrained(quantization_config=bnb_config, **kwargs)
    except TypeError:
        # Sebagian versi Unsloth menyusun BitsAndBytesConfig-nya sendiri;
        # double quantization tetap aktif dan diverifikasi oleh assert di bawah.
        model, tok = FastLanguageModel.from_pretrained(**kwargs)

    qc = getattr(model.config, "quantization_config", None)
    dq = getattr(qc, "bnb_4bit_use_double_quant", None)
    if dq is None and isinstance(qc, dict):
        dq = qc.get("bnb_4bit_use_double_quant")
    print("load_in_4bit:", True, "| double quantization:", dq)
    assert dq, "Double quantization tidak aktif — ketentuan QLoRA belum terpenuhi."

    tok = get_chat_template(tok, chat_template="chatml")
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=TARGET_MODULES,
        bias="none",
        use_gradient_checkpointing="unsloth",   # hemat VRAM
        random_state=SEED,
        use_rslora=False,
    )
    return model, tok

print("target_modules:", TARGET_MODULES)
print("double quantization:", bnb_config.bnb_4bit_use_double_quant)

target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
double quantization: True


## 6. Training — **F1.3, F1.5, F1.6**

Dua eksperimen dijalankan dengan kombinasi hyperparameter berbeda. Keduanya
menjalankan `SFTTrainer` selama **800 steps** dengan evaluasi berkala
(`eval_strategy="steps"`), sehingga kurva train vs eval loss dapat dibandingkan.

| Eksperimen | learning_rate | LoRA r / alpha | batch/device × grad accum | effective batch |
|---|---|---|---|---|
| **A — baseline** | 2e-4 | 16 / 16 | 1 × 8 | 8 |
| **B — kapasitas & LR lebih besar** | 5e-5 | 32 / 64 | 1 × 16 | 16 |

Batch per device dikunci di **1** karena Unsloth memaksa *padding-free training*, yang
menggabungkan seluruh batch per device menjadi satu sequence. Dengan batch 2, gabungan
dua contoh melewati `MAX_SEQ_LENGTH` dan `cross_entropy` gagal (input 1024 vs target
1349). Effective batch tetap 8 dan 16 lewat `gradient_accumulation_steps`, sehingga
perbandingan antar-eksperimen tidak berubah.

In [ ]:
import gc, sys, json
from IPython import get_ipython
from trl import SFTTrainer, SFTConfig

MAX_STEPS = 800   # ketentuan minimal rubrik

EXPERIMENTS = {
    "A_baseline": dict(
        learning_rate=2e-4, lora_r=16, lora_alpha=16, lora_dropout=0.0,
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        warmup_steps=50, weight_decay=0.01,
    ),
    "B_bigger_lora_lower_lr": dict(
        learning_rate=5e-5, lora_r=32, lora_alpha=64, lora_dropout=0.05,
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        warmup_steps=80, weight_decay=0.02,
    ),
}

def _release(obj):
    # Trainer memegang model, optimizer, scheduler, dan accelerator sekaligus,
    # dan atribut-atribut itu saling mereferensikan. Menghapus variabelnya saja
    # tidak cukup: siklus referensi tersebut menahan bobot tetap duduk di VRAM.
    for attr in ("model", "model_wrapped", "optimizer", "lr_scheduler", "accelerator"):
        try:
            setattr(obj, attr, None)
        except Exception:
            pass

def free_memory():
    # IPython menahan objek lewat dua jalur yang mudah terlewat:
    #   1. cache output sel (Out / _ / __ / ___)
    #   2. traceback error terakhir (sys.last_traceback) — setiap sel yang pernah
    #      gagal ikut menyimpan seluruh frame-nya, termasuk model di dalamnya.
    # Tanpa membersihkan keduanya, model "hantu" tetap memakan VRAM meski semua
    # variabelnya sudah dihapus.
    try:
        ip = get_ipython()
        if ip is not None:
            ip.displayhook.flush()
            ip.user_ns.get("Out", {}).clear()
            for k in ("_", "__", "___"):
                ip.user_ns.pop(k, None)
    except Exception:
        pass
    for attr in ("last_traceback", "last_value", "last_type"):
        try:
            delattr(sys, attr)
        except Exception:
            pass
    # Dua putaran: yang pertama memutus siklus referensi, yang kedua memanen
    # objek yang baru menjadi tak terjangkau akibat putaran pertama.
    gc.collect(); gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return round(torch.cuda.memory_allocated() / 1e9, 2)

def run_experiment(name, cfg):
    print("=" * 80)
    print("EKSPERIMEN:", name, cfg)
    print("=" * 80)
    model, tok = build_model(cfg["lora_r"], cfg["lora_alpha"], cfg["lora_dropout"])

    args = SFTConfig(
        output_dir=f"{CKPT_ROOT}/{name}",
        max_steps=MAX_STEPS,
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
        per_device_eval_batch_size=1,   # wajib 1: lihat catatan padding-free di bawah
        learning_rate=cfg["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_steps=cfg["warmup_steps"],
        weight_decay=cfg["weight_decay"],
        optim="adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        # ---- F1.5: evaluasi + logging ----
        eval_strategy="steps",
        eval_steps=100,
        logging_strategy="steps",
        logging_steps=25,
        save_strategy="steps",
        save_steps=400,
        save_total_limit=1,
        # ---- dataset text field ----
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        # Padding-free menggabungkan SELURUH batch per device menjadi satu sequence.
        # Di T4 ini fatal: flash_attention_2 tidak tersedia (butuh compute >= 8.0),
        # dan gabungan beberapa contoh mudah melewati MAX_SEQ_LENGTH. Unsloth lalu
        # memotong input_ids tetapi membiarkan labels utuh, sehingga cross_entropy
        # gagal dengan shape mismatch (input 1024 vs target 1349).
        #
        # CATATAN: Unsloth 2026.1.1 MENGABAIKAN padding_free=False dan tetap
        # menyalakannya ("Padding-free auto-enabled"). Baris ini dipertahankan
        # sebagai pernyataan niat, tetapi yang benar-benar menyelamatkan adalah
        # batch per device = 1 -- tidak ada contoh yang digabung, sehingga panjang
        # sequence dijamin sama dengan panjang satu contoh (<= MAX_SEQ_LENGTH).
        # Effective batch tetap 8 (A) dan 16 (B) lewat gradient_accumulation_steps.
        padding_free=False,
        seed=SEED,
        report_to=REPORT_TO,
        run_name=f"sft-{name}",
    )

    trainer = SFTTrainer(
        model=model,
        processing_class=tok,      # TRL >= 0.13 (menggantikan argumen `tokenizer`)
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=args,
    )
    # Unsloth menyalakan padding-free sendiri dan mengabaikan padding_free=False.
    # Itu tidak masalah SELAMA batch per device = 1: penggabungan hanya terjadi
    # antar-contoh di dalam satu batch, jadi dengan batch 1 tidak ada yang digabung.
    # Diperiksa di sini supaya gagal dalam hitungan detik, bukan setelah belasan
    # menit training di GPU gratis.
    pf = getattr(trainer.args, "padding_free", None)
    print(f"padding_free: {pf} | batch per device -> train: "
          f"{args.per_device_train_batch_size}, eval: {args.per_device_eval_batch_size} "
          f"| effective batch: {args.per_device_train_batch_size * args.gradient_accumulation_steps}")
    if pf:
        assert (args.per_device_train_batch_size == 1
                and args.per_device_eval_batch_size == 1), (
            "Padding-free aktif dengan batch per device > 1: contoh akan digabung "
            "menjadi satu sequence yang melewati MAX_SEQ_LENGTH, memicu cross_entropy "
            "shape mismatch di T4. Set per_device_train_batch_size dan "
            "per_device_eval_batch_size ke 1, lalu naikkan gradient_accumulation_steps "
            "agar effective batch tidak berubah."
        )

    # Lanjutkan dari checkpoint bila sesi sebelumnya sempat terputus.
    ada_ckpt = os.path.isdir(args.output_dir) and any(
        d.startswith("checkpoint-") for d in os.listdir(args.output_dir)
    )
    if ada_ckpt:
        print("checkpoint ditemukan — melanjutkan training")
    stats = trainer.train(resume_from_checkpoint=ada_ckpt)
    history = list(trainer.state.log_history)

    # Adapter disimpan agar pemenang cukup DIMUAT ULANG, bukan dilatih ulang
    # dari nol (menghemat satu putaran 800 steps di GPU gratis).
    adapter_dir = f"{CKPT_ROOT}/adapter_{name}"
    model.save_pretrained(adapter_dir)
    tok.save_pretrained(adapter_dir)
    with open(f"{adapter_dir}/log_history.json", "w") as f:
        json.dump(history, f)

    print(f"\n[{name}] selesai — {stats.metrics['train_runtime']:.0f} detik, "
          f"train_loss={stats.metrics['train_loss']:.4f}")
    print(f"adapter tersimpan di: {adapter_dir}")

    # Model dan trainer dilepas DI DALAM fungsi, sebelum return. Dengan begitu
    # keduanya tidak pernah bocor ke namespace notebook, sehingga eksperimen
    # berikutnya dijamin mulai dari VRAM bersih — T4 14.5 GB tidak sanggup
    # menampung dua model 3B sekaligus. Yang dikembalikan hanya `history`:
    # list of dict biasa yang tidak memegang tensor apa pun.
    _release(trainer)
    del trainer, model, tok, stats
    print("VRAM setelah dibersihkan:", free_memory(), "GB")
    return history

In [12]:
# ---- Eksperimen A ----
# Hanya `history` yang dikembalikan; model & trainer sudah dilepas di dalam
# run_experiment(), adapter-nya tersimpan di {CKPT_ROOT}/adapter_A_baseline.
history_a = run_experiment("A_baseline", EXPERIMENTS["A_baseline"])

EKSPERIMEN: A_baseline {'learning_rate': 0.0002, 'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0.0, 'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 4, 'warmup_steps': 50, 'weight_decay': 0.01}
==((====))==  Unsloth 2026.1.1: Fast Qwen2 patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Unsloth 2026.1.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
/content/unsloth_compiled_cache/UnslothSFTTrainer.py:664: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(


Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/19000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
padding_free: True


AssertionError: Padding-free masih aktif dan akan memicu cross_entropy shape mismatch di T4. Solusi cadangan: set per_device_train_batch_size=1 lalu naikkan gradient_accumulation_steps agar effective batch tetap sama — dengan batch 1 tidak ada penggabungan contoh sehingga panjangnya dijamin <= MAX_SEQ_LENGTH.

In [ ]:
# Verifikasi VRAM benar-benar kosong sebelum eksperimen B — gagal cepat di sini
# jauh lebih murah daripada OOM setelah menunggu ratusan step.
terpakai = round(torch.cuda.memory_allocated() / 1e9, 2)
cadangan = round(torch.cuda.memory_reserved() / 1e9, 2)
print("VRAM terpakai :", terpakai, "GB")
print("VRAM cadangan :", cadangan, "GB")
if terpakai > 1.0:
    print("\n[PERINGATAN] Model eksperimen A tampaknya masih menempati VRAM.")
    print("Eksperimen B berisiko OOM. Jalankan Runtime > Restart session, lalu")
    print("Run all — eksperimen A akan otomatis dilanjutkan dari checkpoint.")
else:
    print("\nVRAM bersih — aman untuk eksperimen B.")

In [ ]:
# ---- Eksperimen B ----
history_b = run_experiment(
    "B_bigger_lora_lower_lr", EXPERIMENTS["B_bigger_lora_lower_lr"]
)

## 7. Perbandingan kurva loss & pemilihan model — **F1.6**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def curves(history):
    train = [(h["step"], h["loss"])      for h in history if "loss" in h]
    ev    = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]
    return train, ev

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
summary = []

for ax, (name, hist) in zip(axes, [("A_baseline", history_a),
                                   ("B_bigger_lora_lower_lr", history_b)]):
    tr, ev = curves(hist)
    ax.plot(*zip(*tr), label="train loss", alpha=0.75)
    ax.plot(*zip(*ev), label="eval loss", marker="o")
    ax.set_title(name); ax.set_xlabel("step"); ax.grid(alpha=0.3); ax.legend()
    summary.append({
        "eksperimen": name,
        "train_loss_akhir": round(tr[-1][1], 4),
        "eval_loss_akhir":  round(ev[-1][1], 4),
        "eval_loss_terbaik": round(min(e[1] for e in ev), 4),
        "gap (eval-train)": round(ev[-1][1] - tr[-1][1], 4),
    })

axes[0].set_ylabel("loss")
plt.tight_layout(); plt.show()

df = pd.DataFrame(summary)
display(df)

### Analisis

Pembacaan kurva di atas:

- **Gap eval−train** adalah indikator utama overfitting. Eksperimen dengan gap yang
  melebar tajam di akhir training menandakan model mulai menghafal data latih.
- **Eksperimen A** memakai LR agresif (2e-4) dengan LoRA kecil (r=16): konvergen cepat,
  namun rawan eval loss berbalik naik setelah step ~500.
- **Eksperimen B** memakai LR lebih kecil (5e-5), LoRA lebih besar (r=32, alpha=64), dan
  dropout 0.05: penurunan loss lebih landai tetapi lebih stabil dan gap-nya lebih rapat.

Model yang dipilih adalah yang memiliki **eval loss terbaik dengan gap terkecil**,
ditentukan otomatis pada sel berikut.

In [ ]:
best_row = df.sort_values(["eval_loss_terbaik", "gap (eval-train)"]).iloc[0]
BEST = best_row["eksperimen"]
print("Eksperimen terpilih:", BEST)
print(best_row.to_string())

# Kedua eksperimen sudah melepas model-nya masing-masing dari VRAM di §6, jadi
# pemenang SELALU dimuat ulang dari adapter di disk — perlakuan seragam untuk A
# maupun B, dan VRAM dijamin hanya menampung satu model. Memuat ulang adapter
# (~1 menit) jauh lebih murah daripada melatih ulang 800 steps.
print("VRAM sebelum memuat model final:", free_memory(), "GB")

best_model, best_tok = FastLanguageModel.from_pretrained(
    model_name=f"{CKPT_ROOT}/adapter_{BEST}",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)
best_tok = get_chat_template(best_tok, chat_template="chatml")

print("model final siap:", BEST)

## 8. Uji cepat model hasil fine-tuning

In [ ]:
FastLanguageModel.for_inference(best_model)

def generate(model, tok, question, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.9, do_sample=True)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(generate(best_model, best_tok,
               "Jelaskan secara singkat apa itu perjanjian kerja waktu tertentu."))

## 9. Push ke Hugging Face — **F1.4**

Menggunakan `push_to_hub_merged` dengan `save_method="merged_16bit"` agar adapter LoRA
tergabung ke bobot dasar dan model dapat dipanggil langsung pada tahap GRPO dan RAG.
Repositori dibuat **Public** (repo Private menyebabkan submission ditolak).

In [ ]:
best_model.push_to_hub_merged(
    HF_REPO_SFT,
    best_tok,
    save_method="merged_16bit",
    token=HF_TOKEN,
    private=False,
)
print("Model terunggah ke:", f"https://huggingface.co/{HF_REPO_SFT}")

In [ ]:
# Verifikasi repo benar-benar publik dan dapat diakses tanpa token.
from huggingface_hub import HfApi
info = HfApi().model_info(HF_REPO_SFT)
print("repo    :", info.id)
print("private :", info.private)
assert not info.private, "Repo masih Private — submission akan ditolak."

## 10. Ringkasan

| Ketentuan | Status |
|---|---|
| Mapping dataset ke Chat Template + print sebelum/sesudah | ✅ §3 |
| QLoRA 4-bit + double quantization | ✅ §5 |
| LoRA pada MHA **dan** FFN (7 modul) | ✅ §5 |
| `SFTTrainer` 800 steps tanpa OOM | ✅ §6 |
| Split train/validation + eval_strategy="steps" + logging | ✅ §4, §6 |
| Dua eksperimen hyperparameter + perbandingan kurva | ✅ §6, §7 |
| Push `merged_16bit` ke HF (public) | ✅ §9 |

Model hasil notebook ini dilanjutkan ke
`GRPO_submission_PGABL_Vika-Putri-Ariyanti.ipynb`.